In [1]:
import torch


In [7]:
import torch
torch.manual_seed(0)

# x: [..., 2d], theta: 与 x 的半维度匹配的角度
B, D = 2, 8
x = torch.randn(B, D, dtype=torch.float32)  # D 必须为偶数
d_half = D // 2

# 正确的 cos/sin：来自同一角度 theta
theta = torch.randn(B, d_half, dtype=torch.float32)  # 这里随便造角度
cos = torch.cos(theta)
sin = torch.sin(theta)

def rope_fwd(x, cos, sin):
    x1, x2 = torch.chunk(x.to(torch.float32), 2, dim=-1)
    y1 = x1 * cos - x2 * sin
    y2 = x2 * cos + x1 * sin
    return torch.cat((y1, y2), dim=-1).to(x.dtype)

def rope_inv(y, cos, sin):
    y1, y2 = torch.chunk(y.to(torch.float32), 2, dim=-1)
    x1 = y1 * cos + y2 * sin
    x2 = y2 * cos - y1 * sin
    return torch.cat((x1, x2), dim=-1).to(y.dtype)

y = rope_fwd(x, cos, sin)
x_rec = rope_inv(y, cos, sin)

print("max abs err:", (x - x_rec).abs().max().item())
print("allclose:", torch.allclose(x, x_rec, rtol=1e-5, atol=1e-6))

max abs err: 1.1920928955078125e-07
allclose: True


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "/local_nvme/liaomengqi/hugingface/hub/models--deepseek-ai--DeepSeek-R1-Distill-Qwen-1.5B/snapshots/ad9f0ae0864d7fbcd1cd905e3c6c5b069cc8b562"
)


In [6]:

t=tokenizer.apply_chat_template(
    [{"role": "user", "content": "introduce yourself\n"},
    {"role": "assistant", "content": "i,m qwen\n"},
    {"role": "user", "content": "Tell me a joke.\n"},],
    tokenize=False,
    add_generation_prompt=True,
)
print(t)

<｜begin▁of▁sentence｜><｜User｜>introduce yourself
<｜Assistant｜>i,m qwen
<｜end▁of▁sentence｜><｜User｜>Tell me a joke.
<｜Assistant｜><think>

